# Supervised Learning Metrics - Regression

Regression models make predictions about continuous values, but how do we know if those predictions are good?

In this notebook, we'll look at the most common ways to evaluate regression models. We'll learn how to measure prediction error and compare models using a few widely used evaluation techniques.

>
> ℹ️ Info about this notebook:
>
> - Goal: lern the most common regression metrics.
> - Dataset: California Housing Dataset
> - Model: Linear Regression
> - Metrics: MAE, MSE, RMSE, and R²
> 


<br>

---

<br>

## Initial Setup

We'll use the **California Housing Dataset** (built into scikit-learn) to train a simple Linear Regression model. The goal is to predict the **median house value** (in units of $100,000) for a district in California based on features like average income, house age, and location.

We'll keep the model simple on purpose — the focus here is on the **metrics**, not the model itself.

In [1]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.datasets import fetch_california_housing
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LinearRegression

In [2]:
# Load the California Housing dataset
housing = fetch_california_housing(as_frame=True)

# Separate features (X) and target (y)
X = housing.data
y = housing.target   # median house value in units of $100,000

print("Dataset shape:", X.shape)
print("\nFeatures:", list(X.columns))
print("\nTarget (first 5 values):", y.values[:5])

Dataset shape: (20640, 8)

Features: ['MedInc', 'HouseAge', 'AveRooms', 'AveBedrms', 'Population', 'AveOccup', 'Latitude', 'Longitude']

Target (first 5 values): [4.526 3.585 3.521 3.413 3.422]


In [3]:
# Split into training and test sets
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# Train a Linear Regression model
model = LinearRegression()
model.fit(X_train, y_train)

# Generate predictions on the test set
y_pred = model.predict(X_test)

print("Model trained successfully.")
print(f"Test set size: {X_test.shape[0]} samples")

Model trained successfully.
Test set size: 4128 samples


<br>

---

<br>

## Common Metrics for Regression

Regression metrics quantify how far off a model's predictions are from the actual values. Different metrics capture different aspects of prediction error — some focus on average error, others penalize large mistakes more heavily, and some express performance as a relative score. Understanding each one helps you choose the right metric for your problem.

<br>

## 1. MAE (Mean Absolute Error)

$$MAE = \frac{1}{n} \sum_{i=1}^{n} |y_i - \hat{y}_i|$$

Where:
- $y_i$ = actual value
- $\hat{y}_i$ = predicted value
- $n$ = number of samples

MAE measures the **average absolute difference** between predictions and actual values. It is expressed in **the same units as the target variable**, making it easy to interpret directly.

A key property of MAE is that it treats all errors equally — it is **robust to outliers** and doesn't punish large errors more than small ones.

In [4]:
from sklearn.metrics import mean_absolute_error

# MAE: average of absolute errors
mae = mean_absolute_error(y_test, y_pred)

print(f"MAE: {mae:.4f}")
print(f"  → On average, predictions are off by ${mae * 100_000:,.0f}")

MAE: 0.5332
  → On average, predictions are off by $53,320


<br>

## 2. MSE (Mean Squared Error)

$$MSE = \frac{1}{n} \sum_{i=1}^{n} (y_i - \hat{y}_i)^2$$

MSE squares each error before averaging. This means **large errors are penalized much more heavily** than small ones — a prediction that is 10 units off contributes 100 times more to MSE than one that is 1 unit off.

The downside: because errors are squared, MSE is expressed in **squared units** of the target, which makes it harder to interpret directly.

> ⚠️ MSE is very sensitive to outliers. A few large prediction errors can inflate MSE significantly.

In [5]:
from sklearn.metrics import mean_squared_error

# MSE: average of squared errors
mse = mean_squared_error(y_test, y_pred)

print(f"MSE: {mse:.4f}")
print(f"  → Units are squared ($100k²), so direct interpretation is less intuitive")

MSE: 0.5559
  → Units are squared ($100k²), so direct interpretation is less intuitive


<br>

## 3. RMSE (Root Mean Squared Error)

$$RMSE = \sqrt{MSE} = \sqrt{\frac{1}{n} \sum_{i=1}^{n} (y_i - \hat{y}_i)^2}$$

RMSE is simply the square root of MSE. Taking the square root brings the metric back to the **same units as the target variable**, which makes it much more interpretable than MSE while still penalizing large errors more than small ones.

**RMSE is one of the most widely used metrics for evaluating regression models** because it combines two desirable properties:
- **Interpretability (same units as target)**: since it is expressed in the same units as the target variable, the error is easy to interpret. For example, if we're predicting house prices in dollars, an RMSE of $10,000 means the model's predictions are typically off by about $10,000.
- **Sensitivity to large errors**: because RMSE is derived from squared errors, larger mistakes have a greater impact on the final score than smaller ones. This makes RMSE particularly useful in applications where large prediction errors are especially costly and should be penalized more heavily.


In [6]:
# RMSE: square root of MSE — brings error back to original units, penalizes large errors more

from sklearn.metrics import root_mean_squared_error

rmse = root_mean_squared_error(y_test, y_pred)

print(f"RMSE: {rmse:.4f}")
print(f"  → Error magnitude (in $100k) with extra weight on big misses")


RMSE: 0.7456
  → Error magnitude (in $100k) with extra weight on big misses


<br>

## 4. R² (R-squared, aka Coefficient of Determination)

$$R^2 = 1 - \frac{SS_{res}}{SS_{tot}}$$

Where:
- $SS_{res} = \sum(y_i - \hat{y}_i)^2$ — sum of squared residuals (model errors)
- $SS_{tot} = \sum(y_i - \bar{y})^2$ — total variance in the target (baseline errors)


Explanation:
- SSres (Residual Sum of Squares): is the total squared error made by the model. It measures how far the model's predictions are from the actual values.
- SStot (Total Sum of Squares): is the total squared error you would get by using a simple baseline that always predicts the average target value. It measures the total variation present in the target variable.

By comparing these two quantities, R² tells us how much of the target's variability is explained by the model:
- **R² = 1.0** → perfect predictions
- **R² = 0.0** → model is no better than predicting the mean
- **R² < 0** → model is worse than the mean baseline

R² measures the **proportion of variance in the target that the model explains**. It answers: *"How much better is our model compared to simply predicting the mean every time?"*

Unlike MAE/MSE/RMSE, R² is **unitless and scale-independent**, making it useful for comparing models across different datasets.

In [7]:
from sklearn.metrics import r2_score

# R²: proportion of variance explained by the model
r2 = r2_score(y_test, y_pred)

print(f"R²: {r2:.4f}")
print(f"  → The model explains {r2 * 100:.1f}% of the variance in house prices")

R²: 0.5758
  → The model explains 57.6% of the variance in house prices


<br>

---

<br>

## Summary

| Metric | Formula | Goal | Units | Outlier Sensitivity | Best Used When |
|---|---|---|---|---|---|
| **MAE** | $\frac{1}{n}\sum\|y_i - \hat{y}_i\|$ | Lower is better | Same as target | Low | You want a robust, easy-to-interpret average error |
| **MSE** | $\frac{1}{n}\sum(y_i - \hat{y}_i)^2$ | Lower is better | Squared units | High | Large errors must be penalized heavily (optimization) |
| **RMSE** | $\sqrt{MSE}$ | Lower is better | Same as target | High | You want MSE's penalty but in interpretable units |
| **R²** | $1 - \frac{SS_{res}}{SS_{tot}}$ | Higher is better | Unitless (0–1) | Moderate | Comparing models or communicating overall fit |

**Key takeaways:**
- Use **RMSE** as your primary reported metric — it's interpretable and penalizes large errors.
- Use **MAE** when your dataset has outliers and you don't want them to dominate the metric.
- Use **R²** to quickly communicate how well the model explains the data.
- In practice, **report multiple metrics together** — no single metric tells the full story.

In [8]:
# All metrics at a glance
print("=" * 40)
print("   Regression Metrics — Summary")
print("=" * 40)
print(f"  MAE:  {mae:.4f}  (${mae * 100_000:,.0f})")
print(f"  MSE:  {mse:.4f}")
print(f"  RMSE: {rmse:.4f}  (${rmse * 100_000:,.0f})")
print(f"  R²:   {r2:.4f}  ({r2 * 100:.1f}% variance explained)")
print("=" * 40)

   Regression Metrics — Summary
  MAE:  0.5332  ($53,320)
  MSE:  0.5559
  RMSE: 0.7456  ($74,558)
  R²:   0.5758  (57.6% variance explained)


<br>

---

<br>

## Combining Metrics in Practice

In practice, it is very common to **combine multiple metrics** to evaluate a model. 

A typical approach is to designate a **primary** and a **secondary** metric:

| Role | Purpose |
|---|---|
| **Primary metric** | - Used to compare and select models (decide which is better)<br>- Main performance indicator reported to stakeholders |
| **Secondary metric** | - Validates the primary metric (acts as a reality check)<br>- Provides additional context for interpretation<br>- Helps detect edge cases or failure modes the primary metric may miss |

<br>

### Common combinations

**1. RMSE + MAE** *(very common in applied ML / production)*
- **RMSE** *(primary)*:
    - Penalizes large errors heavily → useful when big mistakes are especially costly.
- **MAE** *(secondary)*: 
    - Average absolute error in original units → interpretable and robust. 
    - Tells you *"on average, we're off by X."*
- **Why together**: RMSE drives aggressive optimization against large errors + MAE gives a realistic picture of the typical error.

<br>

**2. R² + RMSE** *(common in reporting / statistics / stakeholder communication)*
- **R²** *(primary)*:
    - Answers *"how well does the model fit? (proportion of variance explained by the model)"*  → intuitive for non-technical stakeholders
    - Limitation: it doesn't reflect absolute error size.
- **RMSE** *(secondary)*: 
    - Absolute error magnitude → shows prediction quality in concrete terms. 
    - Why? Two models with the same R² can have very different error scales.
- **Why together**: R² justifies the model to decision-makers + RMSE grounds it in reality (what's the actual error?).